# 06 — Galería provisional y verificación

In [ ]:
from pathlib import Path
import json, shutil, zipfile, hashlib
import torch, numpy as np
from PIL import Image, ImageDraw
from torch import nn
from IPython.display import display
from google.colab import drive
assert torch.cuda.is_available(), "Activa T4 GPU."
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/PuertoPrisma")
POOL = PROJECT / "candidates/stability_lr_d_epoch_015_seed_7070"
IDS = [3,5,7,13,16,24,40,93,120,163]
assert len(IDS)==10 and len(set(IDS))==10
meta=json.loads((POOL / "pool.json").read_text())
records=json.loads((POOL / "nearest_neighbors.json").read_text())
assert meta["generated_count"]==200
OUT = PROJECT / "galeria_provisional"
OUT.mkdir(exist_ok=True)
for folder in ["galeria","vecinos","evidencias"]:
    (OUT/folder).mkdir(exist_ok=True)
print("Exportaremos",IDS,"a",OUT)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, features*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features*8), nn.ReLU(True),
            nn.ConvTranspose2d(features*8, features*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*4), nn.ReLU(True),
            nn.ConvTranspose2d(features*4, features*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features*2), nn.ReLU(True),
            nn.ConvTranspose2d(features*2, features, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features), nn.ReLU(True),
            nn.ConvTranspose2d(features, 3, 4, 2, 1, bias=False), nn.Tanh())
    def forward(self, z):
        return self.net(z)


DEVICE=torch.device("cuda")
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
state=torch.load(POOL / "generator.pt",map_location="cpu",weights_only=True)
G=Generator(state["config"]["latent_dim"],state["config"]["features"]).to(DEVICE).eval()
G.load_state_dict(state["generator"])
z=torch.load(POOL / "noise.pt",map_location="cpu",weights_only=True)
assert len(z)==200
max_error=0
# Mantener los batches originales evita diferencias por cambiar su tamaño.
with torch.inference_mode():
    for start in sorted({((i-1)//32)*32 for i in IDS}):
        generated=((G(z[start:start+32].to(DEVICE)).cpu().clamp(-1,1)+1)*127.5).round().byte()
        for candidate_id in IDS:
            index=candidate_id-1
            if start<=index<start+len(generated):
                regenerated=generated[index-start].permute(1,2,0).numpy()
                with Image.open(POOL / "candidates" / f"candidate_{candidate_id:03d}.png") as image:
                    reference=np.array(image.convert("RGB"))
                error=int(np.abs(regenerated.astype(np.int16)-reference.astype(np.int16)).max())
                max_error=max(max_error,error)
verification=dict(max_pixel_error=max_error,exact_match=max_error==0,
                  method="Same saved vectors, weights and original batch size 32",torch_version=torch.__version__)
(OUT / "evidencias/regeneracion.json").write_text(json.dumps(verification,indent=2))
print("REGENERACIÓN COMPROBADA:",verification)
if max_error>1:
    raise RuntimeError("Diferencia mayor a 1 nivel de píxel: revisar antes de exportar.")
shutil.copy2(POOL / "generator.pt",OUT / "generator.pt")
shutil.copy2(POOL / "noise.pt",OUT / "noise_pool.pt")
torch.save(z[torch.tensor(IDS)-1],OUT / "galeria/vectores_z.pt")
for candidate_id in IDS:
    shutil.copy2(POOL / "candidates" / f"candidate_{candidate_id:03d}.png",OUT / "galeria" / f"personaje_{candidate_id:03d}.png")

In [ ]:
LOCAL=Path("/content/puerto_prisma_export")
LOCAL.mkdir(exist_ok=True)
shutil.copy2(PROJECT / "cartoonset64.zip",LOCAL / "cartoonset64.zip")
assert hashlib.sha256((LOCAL / "cartoonset64.zip").read_bytes()).hexdigest()==meta["data_sha256"]
REAL=LOCAL / "real"
REAL.mkdir(exist_ok=True)
with zipfile.ZipFile(LOCAL / "cartoonset64.zip") as bundle:
    assert all(not Path(name).is_absolute() and ".." not in Path(name).parts for name in bundle.namelist())
    bundle.extractall(REAL)
selected=[]
for candidate_id in IDS:
    row=next(r for r in records if r["candidate_id"]==candidate_id)
    neighbor=REAL / row["nearest_training_file"]
    shutil.copy2(neighbor,OUT / "vecinos" / f"vecino_{candidate_id:03d}.png")
    selected.append(row)
selection=dict(status="provisional_pending_visual_diversity_review",ids=IDS,selected_count=10,
               generated_count=200,selection_rate=0.05,
               criteria="Recognizable faces, fewer artifacts, differences in multiple traits, visual nearest-neighbor review",
               pool_metadata=meta,neighbors=selected,verification=verification,
               limitations="ImageNet features do not certify novelty; repeated archetypes and artifacts remain; IDs 007 and 128 require diversity review")
(OUT / "seleccion.json").write_text(json.dumps(selection,indent=2))
(OUT / "galeria/ids.json").write_text(json.dumps(IDS))
gallery=Image.new("RGB",(5*176,2*208),"#eeeeee")
pairs=Image.new("RGB",(2*368,5*208),"#eeeeee")
gd,pd=ImageDraw.Draw(gallery),ImageDraw.Draw(pairs)
for pos,row in enumerate(selected):
    candidate_id=row["candidate_id"]
    with Image.open(OUT / "galeria" / f"personaje_{candidate_id:03d}.png") as im:
        fake=im.resize((160,160),Image.Resampling.NEAREST)
    with Image.open(OUT / "vecinos" / f"vecino_{candidate_id:03d}.png") as im:
        real=im.convert("RGB").resize((160,160),Image.Resampling.NEAREST)
    x,y=(pos%5)*176,(pos//5)*208
    gallery.paste(fake,(x+8,y+8));gd.text((x+8,y+176),f"ID {candidate_id:03d}",fill="black")
    x,y=(pos%2)*368,(pos//2)*208
    pairs.paste(fake,(x+8,y+24));pairs.paste(real,(x+192,y+24))
    pd.text((x+8,y+4),f"ID {candidate_id:03d} GAN | vecino real",fill="black")
    pd.text((x+8,y+188),f"Distancia coseno: {row['cosine_distance']:.4f}",fill="black")
gallery.save(OUT / "evidencias/galeria_provisional.png")
pairs.save(OUT / "evidencias/vecinos_finalistas.png")
display(gallery)
display(pairs)
print("GALERÍA PROVISIONAL GUARDADA. Selección: 10 de 200 (5 %).")
print("Comparte la rejilla y la salida REGENERACIÓN COMPROBADA. No necesitas entrenar de nuevo.")